## Bước 1 - Tự tính chỉ số Gini


In [ ]:
"""Bước 1: tự tính chỉ số Gini và tự chọn câu hỏi tốt nhất."""

import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text


nho = pd.DataFrame(
    {
        "thu_nhap": [6, 8, 9, 11, 13, 15, 18, 20, 25, 30],
        "co_viec_on_dinh": [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
        "tra_dung_han": [0, 1, 0, 0, 1, 1, 0, 1, 1, 1],
    }
)
y = nho["tra_dung_han"].to_numpy()


def gini(labels):
    """Chỉ số Gini của một nút có hai lớp."""
    if len(labels) == 0:
        return 0.0
    p1 = labels.mean()
    return 1 - p1**2 - (1 - p1) ** 2


def gini_sau_chia(di_trai):
    """Gini trung bình có trọng số của hai nhánh."""
    trai, phai = y[di_trai], y[~di_trai]
    n = len(y)
    return len(trai) / n * gini(trai) + len(phai) / n * gini(phai)


gini_goc = gini(y)
print(f"Gini của nút gốc: {gini_goc:.4f}\n")

thu_nhap = nho["thu_nhap"].to_numpy()
muc = np.sort(np.unique(thu_nhap))
print("Câu hỏi 'thu_nhap <= ngưỡng?'")
print(f"{'ngưỡng':>8} {'Gini sau chia':>15} {'Gini giảm':>12}")
for a, b in zip(muc[:-1], muc[1:]):
    nguong = (a + b) / 2
    gini_chia = gini_sau_chia(thu_nhap <= nguong)
    print(f"{nguong:8.1f} {gini_chia:15.4f} {gini_goc - gini_chia:12.4f}")

viec_lam = nho["co_viec_on_dinh"].to_numpy()
gini_viec = gini_sau_chia(viec_lam == 0)
print("\nCâu hỏi 'có việc ổn định?'")
print(f"Gini sau chia {gini_viec:.4f}, Gini giảm {gini_goc - gini_viec:.4f}")

features = ["thu_nhap", "co_viec_on_dinh"]
tree = DecisionTreeClassifier(max_depth=1, random_state=0).fit(nho[features], y)
print("\nCây một tầng do scikit-learn chọn:")
print(export_text(tree, feature_names=features, show_weights=True))


Gini của nút gốc: 0.4800

Câu hỏi 'thu_nhap <= ngưỡng?'
  ngưỡng   Gini sau chia    Gini giảm
     7.0          0.4000       0.0800
     8.5          0.4750       0.0050
    10.0          0.4190       0.0610
    12.0          0.3167       0.1633
    14.0          0.4000       0.0800
    16.5          0.4500       0.0300
    19.0          0.3429       0.1371
    22.5          0.4000       0.0800
    27.5          0.4444       0.0356

Câu hỏi 'có việc ổn định?'
Gini sau chia 0.4190, Gini giảm 0.0610

Cây một tầng do scikit-learn chọn:
|--- thu_nhap <= 12.00
|   |--- weights: [3.00, 1.00] class: 0
|--- thu_nhap >  12.00
|   |--- weights: [1.00, 5.00] class: 1



## Bước 2 - Cây nhỏ có năm lá


In [ ]:
"""Bước 2: học một cây nhỏ năm lá trên dữ liệu khách vay rồi đọc nó."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
print("Kích thước bảng:", df.shape)
print(df.head())
print(f"\nTỷ lệ khách trả đúng hạn: {df['tra_dung_han'].mean():.2f}\n")

X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

tree = DecisionTreeClassifier(max_leaf_nodes=5, random_state=0).fit(X_train, y_train)
print(f"Độ sâu {tree.get_depth()}, số lá {tree.get_n_leaves()}")
print(export_text(tree, feature_names=list(X.columns), show_weights=True, decimals=3))
print(f"Accuracy tập học     : {accuracy_score(y_train, tree.predict(X_train)):.4f}")
print(f"Accuracy tập kiểm tra: {accuracy_score(y_test, tree.predict(X_test)):.4f}")

new_customers = pd.DataFrame(
    {
        "thu_nhap": [20.0, 10.0],
        "ty_le_no": [0.4, 0.3],
        "co_viec_on_dinh": [1, 0],
        "so_nguoi_phu_thuoc": [2, 1],
    }
)
predictions = tree.predict(new_customers)
probabilities = tree.predict_proba(new_customers)[:, 1]
print()
for i, (prediction, probability) in enumerate(zip(predictions, probabilities), start=1):
    result = "đúng hạn" if prediction == 1 else "trễ hạn"
    print(f"Khách {i}: {result}, xác suất đúng hạn {probability:.2f}")


Kích thước bảng: (600, 5)
   thu_nhap  ty_le_no  co_viec_on_dinh  so_nguoi_phu_thuoc  tra_dung_han
0       9.4      0.18                0                   4             0
1      15.8      0.70                0                   2             1
2       6.1      0.79                0                   0             0
3      19.4      1.13                1                   4             0
4      24.8      0.33                1                   0             1

Tỷ lệ khách trả đúng hạn: 0.49

Độ sâu 3, số lá 5
|--- thu_nhap <= 15.050
|   |--- ty_le_no <= 0.615
|   |   |--- co_viec_on_dinh <= 0.500
|   |   |   |--- weights: [33.000, 6.000] class: 0
|   |   |--- co_viec_on_dinh >  0.500
|   |   |   |--- weights: [35.000, 47.000] class: 1
|   |--- ty_le_no >  0.615
|   |   |--- weights: [114.000, 14.000] class: 0
|--- thu_nhap >  15.050
|   |--- ty_le_no <= 1.020
|   |   |--- weights: [27.000, 145.000] class: 1
|   |--- ty_le_no >  1.020
|   |   |--- weights: [20.000, 9.000] class: 0

Accu

## Bước 3 - Chọn độ sâu của cây


In [ ]:
"""Bước 3: để cây mọc tự do, rồi chọn độ sâu bằng kiểm định chéo."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

free_tree = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
print("Cây mọc tự do")
print(f"  độ sâu {free_tree.get_depth()}, số lá {free_tree.get_n_leaves()}")
print(f"  tập học {accuracy_score(y_train, free_tree.predict(X_train)):.4f}")
print(f"  tập kiểm tra {accuracy_score(y_test, free_tree.predict(X_test)):.4f}\n")

print(f"{'độ sâu':>8} {'tập học':>10} {'kiểm định chéo':>15}")
scores = {}
for depth in range(1, 16):
    model = DecisionTreeClassifier(max_depth=depth, random_state=0)
    scores[depth] = cross_val_score(model, X_train, y_train, cv=5).mean()
    model.fit(X_train, y_train)
    train_accuracy = accuracy_score(y_train, model.predict(X_train))
    print(f"{depth:8d} {train_accuracy:10.4f} {scores[depth]:15.4f}")

best_depth = max(scores, key=scores.get)
best_tree = DecisionTreeClassifier(max_depth=best_depth, random_state=0).fit(X_train, y_train)
print(f"\nĐộ sâu tốt nhất = {best_depth}, cây có {best_tree.get_n_leaves()} lá")
print(f"Accuracy trên tập kiểm tra = {accuracy_score(y_test, best_tree.predict(X_test)):.4f}")


Cây mọc tự do
  độ sâu 13, số lá 92
  tập học 1.0000
  tập kiểm tra 0.7667

  độ sâu    tập học  kiểm định chéo
       1     0.7467          0.7244
       2     0.7711          0.7378
       3     0.8000          0.7511
       4     0.8644          0.8089
       5     0.8978          0.8333
       6     0.9133          0.8156
       7     0.9400          0.8067
       8     0.9600          0.7733
       9     0.9800          0.7800
      10     0.9911          0.7822
      11     0.9933          0.7756
      12     0.9978          0.7778
      13     1.0000          0.7733
      14     1.0000          0.7733
      15     1.0000          0.7733

Độ sâu tốt nhất = 5, cây có 27 lá
Accuracy trên tập kiểm tra = 0.8533


## Bước 4 - Cây không cần chuẩn hóa


In [ ]:
"""Bước 4: kiểm chứng cây không cần chuẩn hóa dữ liệu."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def cay_sau_5():
    return DecisionTreeClassifier(max_depth=5, random_state=0)


original = cay_sau_5().fit(X_train, y_train)
pred_original = original.predict(X_test)

scaler = StandardScaler().fit(X_train)
standardized = cay_sau_5().fit(scaler.transform(X_train), y_train)
pred_standardized = standardized.predict(scaler.transform(X_test))

X_train_thousand, X_test_thousand = X_train.copy(), X_test.copy()
X_train_thousand["thu_nhap"] *= 1000
X_test_thousand["thu_nhap"] *= 1000
thousand = cay_sau_5().fit(X_train_thousand, y_train)
pred_thousand = thousand.predict(X_test_thousand)

print("Accuracy trên tập kiểm tra")
print(f"  dữ liệu gốc          : {accuracy_score(y_test, pred_original):.4f}")
print(f"  đã chuẩn hóa         : {accuracy_score(y_test, pred_standardized):.4f}")
print(f"  thu nhập nghìn đồng  : {accuracy_score(y_test, pred_thousand):.4f}")
print("\nSố khách bị đoán khác cây gốc")
print(f"  đã chuẩn hóa         : {(pred_standardized != pred_original).sum()}")
print(f"  thu nhập nghìn đồng  : {(pred_thousand != pred_original).sum()}")
print("\nNgưỡng của câu hỏi gốc")
print(f"  dữ liệu gốc          : {original.tree_.threshold[0]:.2f}")
print(f"  thu nhập nghìn đồng  : {thousand.tree_.threshold[0]:.2f}")


Accuracy trên tập kiểm tra
  dữ liệu gốc          : 0.8533
  đã chuẩn hóa         : 0.8533
  thu nhập nghìn đồng  : 0.8533

Số khách bị đoán khác cây gốc
  đã chuẩn hóa         : 0
  thu nhập nghìn đồng  : 0

Ngưỡng của câu hỏi gốc
  dữ liệu gốc          : 15.05
  thu nhập nghìn đồng  : 15050.00


## Bước 5 - Rừng ngẫu nhiên


In [ ]:
"""Bước 5: trồng rừng ngẫu nhiên và so sánh với một cây."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

rng = np.random.default_rng(0)
n = len(X_train)
indices = rng.integers(0, n, n)
different = len(np.unique(indices))
print(f"Bốc {n} lần, có {different} khách khác nhau ({different / n:.0%})\n")

forest = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
print("Rừng 200 cây")
print(f"  tập học     : {accuracy_score(y_train, forest.predict(X_train)):.4f}")
print(f"  tập kiểm tra: {accuracy_score(y_test, forest.predict(X_test)):.4f}")

print("\nSố cây và accuracy tập kiểm tra")
for number in [1, 10, 50, 200, 400]:
    model = RandomForestClassifier(n_estimators=number, random_state=0).fit(X_train, y_train)
    print(f"  {number:4d}: {accuracy_score(y_test, model.predict(X_test)):.4f}")

models = {
    "Cây mọc tự do": DecisionTreeClassifier(random_state=0),
    "Cây sâu 5": DecisionTreeClassifier(max_depth=5, random_state=0),
    "Rừng 200 cây": forest,
}
print("\nMô hình          tập học  tập kiểm tra")
for name, model in models.items():
    model.fit(X_train, y_train)
    train_accuracy = accuracy_score(y_train, model.predict(X_train))
    test_accuracy = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:<16} {train_accuracy:.4f}    {test_accuracy:.4f}")


Bốc 450 lần, có 286 khách khác nhau (64%)

Rừng 200 cây
  tập học     : 1.0000
  tập kiểm tra: 0.9000

Số cây và accuracy tập kiểm tra
     1: 0.7933
    10: 0.8467
    50: 0.8733
   200: 0.9000
   400: 0.9000

Mô hình          tập học  tập kiểm tra
Cây mọc tự do    1.0000    0.7667
Cây sâu 5        0.8978    0.8533
Rừng 200 cây     1.0000    0.9000


## Bước 6 - Độ quan trọng của đặc trưng


In [ ]:
"""Bước 6: đọc độ quan trọng của đặc trưng và thấy giới hạn của nó."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "khach_vay.csv")
X = df.drop(columns="tra_dung_han")
y = df["tra_dung_han"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

forest = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
importance = pd.Series(forest.feature_importances_, index=X.columns)
print("Độ quan trọng, bốn cột thật")
print(importance.sort_values(ascending=False).round(3).to_string())
print("Tổng:", round(importance.sum(), 3))

rng = np.random.default_rng(0)
X_train_random = X_train.copy()
X_train_random["so_ngau_nhien"] = rng.random(len(X_train))
forest_random = RandomForestClassifier(n_estimators=200, random_state=0).fit(
    X_train_random, y_train
)
importance_random = pd.Series(
    forest_random.feature_importances_, index=X_train_random.columns
)
print("\nĐộ quan trọng khi thêm một cột ngẫu nhiên")
print(importance_random.sort_values(ascending=False).round(3).to_string())


Độ quan trọng, bốn cột thật
thu_nhap              0.479
ty_le_no              0.339
so_nguoi_phu_thuoc    0.093
co_viec_on_dinh       0.088
Tổng: 1.0

Độ quan trọng khi thêm một cột ngẫu nhiên
thu_nhap              0.427
ty_le_no              0.270
so_ngau_nhien         0.150
co_viec_on_dinh       0.077
so_nguoi_phu_thuoc    0.076


## Bước 7 - Cây cho bài toán hồi quy


In [ ]:
"""Bước 7: cây quyết định cho bài toán hồi quy trên dữ liệu giá nhà."""

from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, export_text


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree = DecisionTreeRegressor(max_depth=3, random_state=0).fit(X_train, y_train)
linear = LinearRegression().fit(X_train, y_train)

print(f"Cây hồi quy sâu 3 có {tree.get_n_leaves()} lá")
print(export_text(tree, feature_names=["dien_tich"], decimals=1))
mae_tree = mean_absolute_error(y_test, tree.predict(X_test))
mae_linear = mean_absolute_error(y_test, linear.predict(X_test))
print("MAE trên tập kiểm tra (tỷ đồng)")
print(f"  Cây sâu 3   : {mae_tree:.4f}")
print(f"  Đường thẳng : {mae_linear:.4f}")

new_houses = pd.DataFrame({"dien_tich": [60.0, 100.0, 150.0, 250.0]})
print("\nDự đoán cho bốn căn mới (tỷ đồng)")
print(f"{'diện tích':>10} {'cây':>8} {'đường thẳng':>12}")
for area, tree_price, linear_price in zip(
    new_houses["dien_tich"], tree.predict(new_houses), linear.predict(new_houses)
):
    print(f"{area:7.0f} m2 {tree_price:8.2f} {linear_price:12.2f}")

print("\nCây không ngoại suy: các diện tích vượt ngưỡng cuối cùng đều rơi vào cùng một lá.")


Cây hồi quy sâu 3 có 8 lá
|--- dien_tich <= 73.9
|   |--- dien_tich <= 64.7
|   |   |--- dien_tich <= 50.5
|   |   |   |--- value: [4.2]
|   |   |--- dien_tich >  50.5
|   |   |   |--- value: [4.9]
|   |--- dien_tich >  64.7
|   |   |--- dien_tich <= 68.5
|   |   |   |--- value: [5.3]
|   |   |--- dien_tich >  68.5
|   |   |   |--- value: [5.9]
|--- dien_tich >  73.9
|   |--- dien_tich <= 93.8
|   |   |--- dien_tich <= 82.5
|   |   |   |--- value: [6.8]
|   |   |--- dien_tich >  82.5
|   |   |   |--- value: [7.4]
|   |--- dien_tich >  93.8
|   |   |--- dien_tich <= 107.7
|   |   |   |--- value: [8.3]
|   |   |--- dien_tich >  107.7
|   |   |   |--- value: [9.4]

MAE trên tập kiểm tra (tỷ đồng)
  Cây sâu 3   : 0.5039
  Đường thẳng : 0.2578

Dự đoán cho bốn căn mới (tỷ đồng)
 diện tích      cây  đường thẳng
     60 m2     4.92         5.12
    100 m2     8.32         8.24
    150 m2     9.40        12.13
    250 m2     9.40        19.91

Cây không ngoại suy: các diện tích vượt ngưỡng cuố